# FLUX.1-dev GGUF Q8_0 — 双卡双开模板
权重来自私有数据集 `lei1237/flux1-dev-q8-gguf`（Add Data 挂载），零下载。改顶部参数即可复用。

In [ ]:
# ===== 参数区（只改这里）=====
PROMPTS = [
    "G.E.M. Deng Ziqi, famous Chinese pop singer, singing passionately on a beach at golden sunset, holding a microphone, ocean waves and palm trees in the background, warm golden hour light, photorealistic portrait, 85mm lens, ultra detailed",
    "G.E.M. Deng Ziqi, famous Chinese pop singer, performing live on a concert stage, holding a microphone, dramatic purple and blue stage lights, cheering crowd silhouettes, photorealistic, ultra detailed",
]
SEEDS     = [42, 43]        # 与 PROMPTS 一一对应；单开就只留一条
STEPS     = 28
GUIDANCE  = 3.5
WIDTH = HEIGHT = 1024
STAGGER_S = 60              # 双开时第二个进程的启动延迟（错开加载内存峰值）

✅ 标准路线（用户 2026-11-09 拍板）：挂载数据集 → **整包拷到本地盘**（掐表）→ 全部组件从本地加载 → 双开出图。编码已关梯度（开梯度会 OOM）。

In [ ]:
# ===== 环境与路径发现 =====
import subprocess, sys, glob, os, json, time, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gguf==0.19.0"], check=True)
import gguf  # 铁律：必须先于 diffusers 导入，否则 GGUF 可用标志被缓存为 False
import torch
from diffusers import FluxTransformer2DModel, FluxPipeline, AutoencoderKL, FlowMatchEulerDiscreteScheduler

cands = glob.glob("/kaggle/input/datasets/**/flux1-dev*.gguf", recursive=True) + glob.glob("/kaggle/input/**/*.gguf", recursive=True)
assert cands, "数据集未挂载或找不到 GGUF 文件：先 Add Data 添加 lei1237/flux1-dev-q8-gguf"
GGUF_PATH = cands[0]
DS = os.path.dirname(GGUF_PATH)
print("GGUF:", GGUF_PATH)
print("DS  :", DS)
print(sorted(os.listdir(DS)))
LOCAL = "/kaggle/working/ds_local_full"
_t0 = time.time()
shutil.copytree(DS, LOCAL, dirs_exist_ok=True)
print(f"整包拷本地: {time.time()-_t0:.1f}s -> {LOCAL}")
DS = LOCAL
GGUF_PATH = glob.glob(DS + "/*.gguf")[0]
print("本地 GGUF:", GGUF_PATH)
print("GPUs:", torch.cuda.device_count())

In [ ]:
# ===== 编码全部提示词（编码器用完即卸）=====
from transformers import CLIPTextModel, CLIPTokenizer, T5EncoderModel, T5TokenizerFast

tok1 = CLIPTokenizer.from_pretrained(DS, subfolder="tokenizer")
tok2 = T5TokenizerFast.from_pretrained(DS, subfolder="tokenizer_2")
te1 = CLIPTextModel.from_pretrained(DS, subfolder="text_encoder", torch_dtype=torch.float16).to("cuda:0")
te2 = T5EncoderModel.from_pretrained(DS, subfolder="text_encoder_2", torch_dtype=torch.float16).to("cuda:0")

torch.set_grad_enabled(False)  # 实测修复：编码开梯度会 OOM
def encode(prompt):
    ids1 = tok1(prompt, padding="max_length", max_length=77, truncation=True, return_tensors="pt").input_ids.to("cuda:0")
    pooled = te1(ids1).pooler_output
    ids2 = tok2(prompt, padding="max_length", max_length=512, truncation=True, return_tensors="pt").input_ids.to("cuda:0")
    emb = te2(ids2).last_hidden_state
    return emb.cpu(), pooled.cpu()

os.makedirs("/kaggle/working/embeds", exist_ok=True)
for i, p in enumerate(PROMPTS):
    emb, pooled = encode(p)
    torch.save({"emb": emb, "pooled": pooled}, f"/kaggle/working/embeds/emb_{i}.pt")
    print(i, emb.shape, pooled.shape)
del te1, te2
torch.cuda.empty_cache()
print("编码完成，编码器已卸载")

In [ ]:
# ===== worker 脚本（每张卡一个独立进程）=====
worker = r"""
import sys, time, json
import gguf
import torch
from diffusers import FluxTransformer2DModel, FluxPipeline, AutoencoderKL, FlowMatchEulerDiscreteScheduler
from diffusers import GGUFQuantizationConfig

idx   = int(sys.argv[1])
ds    = sys.argv[2]
gguf  = sys.argv[3]
dev   = "cuda:0"  # CUDA_VISIBLE_DEVICES 已把本进程的卡映射为 0
t0 = time.time()
transformer = FluxTransformer2DModel.from_single_file(gguf, config=f"{ds}/transformer", quantization_config=GGUFQuantizationConfig(compute_dtype=torch.float16))
transformer = transformer.to(dev)  # 量化模型不可转精度，compute_dtype 已在配置中设定
vae = AutoencoderKL.from_pretrained(ds, subfolder="vae", torch_dtype=torch.float16).to(dev)
sched = FlowMatchEulerDiscreteScheduler.from_pretrained(ds, subfolder="scheduler")
pipe = FluxPipeline(scheduler=sched, vae=vae, text_encoder=None, tokenizer=None,
                    text_encoder_2=None, tokenizer_2=None, transformer=transformer)
t_load = time.time() - t0
d = torch.load(f"/kaggle/working/embeds/emb_{idx}.pt")
g = torch.Generator(device=dev).manual_seed(SEED)
t1 = time.time()
out = pipe(prompt_embeds=d["emb"].to(dev, torch.float16),
           pooled_prompt_embeds=d["pooled"].to(dev, torch.float16),
           width=W, height=H, num_inference_steps=STEPS, guidance_scale=GUIDANCE,
           generator=g).images[0]
t_gen = time.time() - t1
out.save(f"/kaggle/working/out_{idx}.png")
print(json.dumps({"idx": idx, "load_s": round(t_load,1), "gen_s": round(t_gen,1),
                  "peak_gb": round(torch.cuda.max_memory_allocated()/2**30, 2)}))
"""
worker = worker.replace("SEED", "int(sys.argv[4])").replace("W", "1024").replace("H", "1024").replace("STEPS", "28").replace("GUIDANCE", "3.5")
open("/kaggle/working/worker.py", "w").write(worker)
print("worker.py 已写入")

In [ ]:
# ===== 双开启动 + 计时汇总 =====
import subprocess, sys, os, time, json
envs = []
procs = []
t_start = time.time()
for i in range(len(PROMPTS)):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
    p = subprocess.Popen([sys.executable, "/kaggle/working/worker.py", str(i), DS, GGUF_PATH, str(SEEDS[i])],
                         env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    procs.append(p)
    if i + 1 < len(PROMPTS):
        time.sleep(STAGGER_S)  # 错峰加载，避开 RAM 峰值
for p in procs:
    print(p.communicate()[0])
wall = time.time() - t_start
print(f"总墙钟: {wall:.1f}s | 每图平均: {wall/len(PROMPTS):.1f}s | 单卡基线 172.8s")

In [ ]:
from IPython.display import Image, display
for i in range(len(PROMPTS)):
    display(Image(filename=f"/kaggle/working/out_{i}.png"))